# PyTorch Lightning baseline template

by Andrés Muñoz-Jaramillo

This notebook is meant to act as a template to train and use a simple regression model to define a baseline that can be compared with a DS application.

It focuses on the concept of defining a PyTorch model, a PyTorch lightning training loop and the deffinition of metrics of performance.

This notebook assumes familiarity with the concepts of datasets and dataloaders contained in the **_0_dataset_dataloader_template.ipynb_**

## Set your cuda visible device

**IMPORTANT:** Since we are sharing resources, please make sure that the cuda visible device you put here is the one assigned to your team and your machine.   

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. It is what lets training.deterministic work without a
# cuBLAS warning on every run. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
from torch.utils.data import DataLoader

import torch
import yaml

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the wokshop_infrastructure folder.
sys.path.append("../../")
 
# Append Surya path. May need to be modified if the folder structure changes.
# It gives the notebook access to surya's release code.

from workshop_infrastructure.utils import build_scalers  # Data scaling utilities for Surya stacks

torch.set_float32_matmul_precision('medium')



## Load configuration

Surya was designed to read a configuration file that defines many aspects of the model
including the data it uses we use this config file to set default values that do not
need to be modified, but also to define values specific to our downstream application

In [4]:
# The config is the single source of truth. load_flare_config() parses it into a typed
# object, exactly as the training script 3_finetune_template_1D.py does, so the same YAML
# behaves identically here and in production. Notebook 0 walks through what it contains.
from downstream_apps.filament_kyle.configs import load_filament_config

cfg = load_filament_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")


Loaded config for job: filament_characterization


## Download assets

The config says where the assets belong, so it is loaded first. `ensure_assets()` fetches only what is missing from HuggingFace, so re-running this is free.


In [5]:
# One implementation, shared by the notebooks, the training script and the
# download_*.sh wrappers: workshop_infrastructure/assets.py.
# The linear baseline needs no backbone, so skip the 1.8 GB weights.
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers"])

# Now that scalers.yaml is guaranteed to be on disk, load it. build_scalers()
# accepts the resolved path directly.
scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")


Loaded scalers for 13 channels.


## Define Downstream (DS) datasets

This child class takes as input all expected HelioFM parameters, plus additonal parameters relevant to the downstream application.  Here we focus in particular to the DS index and parameters necessary to combine it with the HelioFM index.

Another important component of creating a dataset class for your DS is normalization.  Here we use a log normalization on xray flux that will act as the output target.  Making log10(xray_flux) strictly positive and having 66% of its values between 0 and 1

In this case we will define both a training and a validation dataset using the indices pointed at in the config

**_Important:  In this notebook we sets max_number_of_samples=6 to potentially avoid going through the whole dataset as we explore it.  Keep in mind this for the future in case the database seems smaller than you expect_**


In [6]:
from downstream_apps.filament_kyle.datasets.filament_dataset import FilamentDataset

In [7]:
# build_helio_dataloaders() constructs the train and validation datasets and wraps them
# in DataLoaders. It fills in every generic argument (channels, temporal sampling, S3
# access, worker settings) from the config — see notebook 0 for what that block looks
# like written out. Only the flare-specific arguments are passed here, which is exactly
# the list you replace when you fork the template.
#
# It also handles two details that are easy to get wrong by hand: the validation set gets
# phase="val" (no random channel masking or flips), and only the training loader shuffles.
from workshop_infrastructure.datasets.builders import build_helio_dataloaders

train_data_loader, val_data_loader = build_helio_dataloaders(
    cfg,
    FilamentDataset,
    scalers=scalers,
    num_workers=4,          # fewer workers than the script: notebooks start faster
    #### Downstream (DS) specific parameters
    return_surya_stack=True,
    max_number_of_samples=6,
    filament_index_path=cfg.data.filament_index_path,
    ds_time_column=cfg.data.ds_time_column,
    ds_time_tolerance=cfg.data.ds_time_tolerance,
    ds_match_direction=cfg.data.ds_match_direction,
)

batch_size = cfg.batch_size
print(f"train: {len(train_data_loader.dataset)} samples | "
      f"val: {len(val_data_loader.dataset)} samples | batch_size: {batch_size}")


ValueError: No intersection between Surya and DS indices

Training and validation get separate datasets and dataloaders. They differ only in the index they read and in `phase`: `phase="val"` turns off the random channel masking and vertical flips used for training augmentation.

The loaders use `multiprocessing_context="spawn"` — the dataset holds an S3 client that does not survive `fork`, and spawn also avoids lockups in shared environments.


In [ ]:
# Inspect a single batch to confirm shapes before training.
batch = next(iter(train_data_loader))
print({k: (tuple(v.shape) if hasattr(v, "shape") else type(v).__name__) for k, v in batch.items()})


## Define simple baseline model

Defining a simple baseline is important to understand what value is bringing the AI model to the problem.  

It is always very good to have a very simple baseline model.  Ideally one that cannot overfit the data.  This is a very good way of really measuring the value added of complex models.   Classical machine learning excels here:

- Regressions and logistic regressions.
- Climatological averages.
- Persistance.
- Simple transformations.

Simple models avoid excesively optimistic assessments of the capatiblities of a complex models and for many problems are actually remarkably hard to beat.

In this example we define a simple regression acting on the intensity of each channel.  Note that we invert the normalization to deal with strictly positive quantities.  As with the dataset we will be importing the model from a module so that we can use it within training scripts later on.

In [ ]:
from downstream_apps.template.models.simple_baseline import RegressionFlareModel

We can now test that this model manipulates a batch as expected and returns an estimate of flare intensity

Note that the simple regression model definition requires knowing the number of channels and timesteps so here we pull that information from the configuration intializing the model.

In [ ]:
n_input_timestamps = cfg.model.time_embedding.time_dim
n_channels = len(cfg.data.channels)

# RegressionFlareModel only needs the flattened input dimension.
# It expects 'ts' in signum-log space — use destandardize_channels() before calling forward().
model = RegressionFlareModel(n_input_timestamps * n_channels)


Now we can pass the input stack 'ts' to the model to transform it into our regression output.   Note that since this model has not been trained and was initalized randomly.  The output here has no real meaning.  It only acts as a test that our model forward doesn't have dimension problems.

Dimension problemns are the dominant source of error in this kind of work.

Note that our output has now the size of our batch.

In [ ]:
from downstream_apps.template.models.simple_baseline import destandardize_channels

batch = next(iter(train_data_loader))

# RegressionFlareModel works in signum-log space, not normalized space.
# destandardize_channels undoes the per-channel z-score but KEEPS the log compression —
# raw DN values span too many orders of magnitude to be good features for one linear layer.
# (For true physical units, use train_dataset.inverse_transform_data() instead. See the
#  "THE THREE SPACES" block in workshop_infrastructure/datasets/helio.py.)
batch_logspace = destandardize_channels(batch, channel_order=cfg.data.channels, scalers=scalers)
output = model.forward(batch_logspace)[:, 0]  # Get rid of singleton dimension
output


## Define your metrics

Metrics are a very important part of training AI models.   They provide your models with the quantitification of error, which in turn shifts the weights towards better pefrorming models.  They also provide a way for you to monitor performance, identify overfitting, and quantify value added. 

We now initialize the metrics class which allows you to control what metrics do you want to use as "loss" (i.e. the metrics that backpropagate through your model) and which ones for monitoring performance.  As with other components, this takes the form of a loaded module that can be later use in a training script

In [ ]:
from downstream_apps.template.metrics.template_metrics import FlareMetrics

In [ ]:
train_loss_metrics = FlareMetrics("train_loss")
# val_loss is the quantity logged as "val_loss" and used to pick the best checkpoint.
# It defaults to the same MSE as train_loss — override FlareMetrics.val_loss to change it.
val_loss_metrics = FlareMetrics("val_loss")
train_evaluation_metrics = FlareMetrics("train_metrics")
# Reported only: val_metrics do NOT influence checkpoint selection.
validation_evaluation_metrics = FlareMetrics("val_metrics")

Now they can be evaluated in our model's output and our ground truth.   First the loss that actually will backpropagate, in this case Mean Squared Errror

In [ ]:
train_loss_metrics(output, batch["forecast"])

Then a training evaluation that will not backpropagate and inform our model, but that we can keep an eye on. Note that reporting lots of metrics during training will slow the training process.  I'm including it her as an example, but oftentimes is better to put the diagnostics only in the validation evaluation metrics.

Here we are caclulating the Root Relative Squared Error https://lightning.ai/docs/torchmetrics/stable/regression/rse.html 

A value below one means the prediction is better than predicting the average.  It is unlikely that this metric will be lower than one with a randomly initialized model

In [ ]:
train_evaluation_metrics(output, batch["forecast"])

In the validation evaluation metrics we report both MSE and RRSE

In [ ]:
validation_evaluation_metrics(output, batch["forecast"])

## Define your PyTorch ligthning module

In this workshop we will use PyTorch lightning to train our models.  PyTorch lighting reduces the amount of code required to implement a training loop in comparison to PyTorch (at the expense of control and versatility).  

Opening the FlareLightningModule shows a simple Lightning model implementation.  It consists of:

- An initialization of the class (metrics, model, and learning rate).
- The forward code that runs evaluation of the model.
- Training and validation steps.
- Configuration of optimizers.

In [ ]:
from downstream_apps.template.lightning_modules.pl_simple_baseline import FlareLightningModule

## Set your global seeds

Since training AI models generally uses stochastic gradient descent, it is a good idea to fix your random seeds so that your training exercise is reproducible.    

In [ ]:
L.seed_everything(42, workers=True)

## Intialize Lightning module

Now we properly initalize the Lightning module to enable training, including passing the dictionary of metrics

In [ ]:
from functools import partial

metrics = {
    'train_loss': train_loss_metrics,
    'val_loss': val_loss_metrics,
    'train_metrics': train_evaluation_metrics,
    'val_metrics': validation_evaluation_metrics,
}

# Wire up the inverse transform so FlareLightningModule applies it before every model call.
preprocess_fn = partial(
    destandardize_channels,
    channel_order=cfg.data.channels,
    scalers=scalers,
)

lit_model = FlareLightningModule(
    model, metrics, lr=cfg.learning_rate, batch_size=batch_size, preprocess_fn=preprocess_fn
)


## Logging

In order to properly compare experiments against each other, it is very useful to log evaluation metrics in a place where they can be compared against other training runs.  In this workshop we will use Weights and Biases (WandB). 

The first time you run WandB in a machine it will ask you to login to WandB.  You should have received an invitation to our project.  In order to login you must:

- Select option 2 (existing account).   In VScode the dialog opens a box at the top of your screen.
- Click on get API Key (this will open a browser).
- Generate API Key.
- Paste it in the dialog box at the top of your VSCode

In [ ]:
project_name = cfg.wandb_project
run_name = "baseline_experiment_1"  # give your run a descriptive name

wandb_logger = WandbLogger(
    entity=cfg.wandb_entity,  # set wandb_entity in the config; null = personal account
    project=project_name,
    name=run_name,
    log_model=False,
    save_dir="./wandb/wandb_tmp",
)

csv_logger = CSVLogger("runs", name=project_name)


## Initialize trainer

With the loggers done, now the trainer needs to be defined.  The trainer defines several properties of your training run. Here we define:

- The max number of epochs (one epoch represents your model seeing your entire training dataset).
- Define where the training run will take place (auto uses the GPU if possible, if not, CPU).
- The loggers.
- The callbacks (here we save the model with the lowest validation loss).
- Logging frequency (because we are working with a small dataset it needs to be small).

In [ ]:
max_epochs = 2

# -------------------------------------------------------------------------
# Trainer
# -------------------------------------------------------------------------
trainer = L.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices="auto",
    logger=[wandb_logger, csv_logger],
    callbacks=[
        ModelCheckpoint(
            monitor="val_loss",
            mode="min",
            save_top_k=1,
        )
    ],
    log_every_n_steps=2,
)

## Fit the model

Finally we fit the model.  We pass the Lighting module, and our dataloaders.

In [ ]:
trainer.fit(lit_model, train_data_loader, val_data_loader)

## Conclusion

With this we have now integrated our dataset, dataloaders, metrics, and baseline into an end-2-end training loop.  The next step is to substitute the simple model with Surya.

## Baselines: coin-flip and calibrated Martin's Rule

Before comparing against Surya, we need a baseline number worth beating. The classic **Martin's Rule** (hemisphere predicts chirality, ~90% in the textbook) is *not* used directly here: this catalog was deliberately curated to include events that violate it, so scoring against the textbook rate would be an unfair comparison point — it could make Surya look artificially better or worse depending on how the exceptions happen to land.

Instead we use two baselines that don't assume the textbook rate:

1. **Coin-flip** — random 50/50 guessing. With only 11 (soon 45) events, a single draw is too noisy to be a stable number, so we run a Monte Carlo simulation and report the resulting accuracy distribution.
2. **Calibrated Martin's Rule** — instead of assuming ~90%, estimate `P(chirality | hemisphere)` directly from this catalog. Evaluated via **leave-one-out cross-validation**: for each event, the hemisphere-conditioned rate is estimated from the other events only, so no event is ever used to predict itself.

Both baselines only need the chirality label and the `hemisphere` passthrough field from `FilamentDataset` — no Surya imagery, no model.

In [ ]:
import numpy as np

cfg = load_filament_config("./configs/config_script.yaml")

# train_data_path/valid_data_path are overridden here in-memory only — config_script.yaml
# still points at the generic shared Surya indices, which don't cover 2012 (9 of these 11
# events fall in 2012). This points instead at the small index built specifically for this
# catalog (see data/surya_index_filament_events.csv).
cfg.data.train_data_path = "./data/surya_index_filament_events.csv"
cfg.data.valid_data_path = "./data/surya_index_filament_events.csv"

baseline_train_loader, baseline_val_loader = build_helio_dataloaders(
    cfg,
    FilamentDataset,
    scalers=scalers,
    num_workers=0,
    return_surya_stack=False,   # neither baseline looks at the Surya imagery
    filament_index_path=cfg.data.filament_index_path,
    ds_time_column=cfg.data.ds_time_column,
    ds_time_tolerance=cfg.data.ds_time_tolerance,
    ds_match_direction=cfg.data.ds_match_direction,
)

baseline_dataset = baseline_train_loader.dataset
n_events = len(baseline_dataset)
labels = np.array([baseline_dataset[i]["forecast"] for i in range(n_events)])
hemispheres = np.array([baseline_dataset[i]["hemisphere"] for i in range(n_events)])

print(f"{n_events} catalog events matched to a Surya frame")
print(f"Chirality counts  — dextral (0): {(labels == 0).sum()}, sinistral (1): {(labels == 1).sum()}")
print(f"Hemisphere counts — south (0): {(hemispheres == 0).sum()}, north (1): {(hemispheres == 1).sum()}")


### Coin-flip baseline (Monte Carlo)

A single set of random 50/50 guesses over 11 events is too noisy to be a meaningful number on its own — one lucky draw could land at 70%+ accuracy by chance. Instead we simulate many trials and report the resulting distribution; the mean is the number to compare Surya against, and the spread shows how much of any single accuracy score is just noise at this sample size.

In [ ]:
rng = np.random.default_rng(42)  # matches the seed used for the Lightning run above
n_trials = 100_000

coin_flip_predictions = rng.integers(0, 2, size=(n_trials, n_events))
coin_flip_accuracies = (coin_flip_predictions == labels).mean(axis=1)

print(f"Coin-flip baseline over {n_trials:,} trials on {n_events} events:")
print(f"  mean accuracy:       {coin_flip_accuracies.mean():.3f}")
print(f"  std accuracy:        {coin_flip_accuracies.std():.3f}")
print(f"  5th-95th percentile: {np.percentile(coin_flip_accuracies, 5):.3f} - {np.percentile(coin_flip_accuracies, 95):.3f}")


### Calibrated Martin's Rule (leave-one-out)

Rather than assume the textbook ~90% hemisphere-to-chirality accuracy, we estimate it from this catalog directly: for each event, take the majority chirality among all *other* events sharing its hemisphere, and predict that majority label. This is leave-one-out — an event never contributes to its own prediction, so the accuracy isn't inflated by fitting and scoring on the same data. Ties, or a fold where no other event shares the held-out hemisphere, fall back to the overall majority chirality among the remaining events.

In [ ]:
def majority_label(values: np.ndarray) -> float:
    """Majority vote among 0/1 labels; ties resolve to dextral (0)."""
    return 1.0 if values.mean() > 0.5 else 0.0


def loo_calibrated_martins_rule(labels: np.ndarray, hemispheres: np.ndarray) -> np.ndarray:
    n = len(labels)
    predictions = np.empty(n)
    for i in range(n):
        others = np.arange(n) != i
        train_labels, train_hemispheres = labels[others], hemispheres[others]
        same_hemisphere = train_hemispheres == hemispheres[i]
        # Fall back to the overall majority class if no other event shares this
        # hemisphere in the held-out fold.
        pool = train_labels[same_hemisphere] if same_hemisphere.any() else train_labels
        predictions[i] = majority_label(pool)
    return predictions


martins_rule_predictions = loo_calibrated_martins_rule(labels, hemispheres)
martins_rule_correct = martins_rule_predictions == labels
martins_rule_accuracy = martins_rule_correct.mean()

print(f"Calibrated Martin's Rule (leave-one-out) accuracy: "
      f"{martins_rule_accuracy:.3f} ({int(martins_rule_correct.sum())}/{n_events})")


### Baseline summary

These are the two numbers Surya needs to beat — not the textbook Martin's Rule accuracy, which this catalog is specifically designed to violate in places.

In [ ]:
print("Baseline summary")
print(f"  Coin-flip (Monte Carlo mean):    {coin_flip_accuracies.mean():.3f}")
print(f"  Calibrated Martin's Rule (LOO):  {martins_rule_accuracy:.3f}")
